# Microsoft Agent Framework 실습 노트북
## 고등학생을 위한 AI 에이전트 입문

이 노트북에서는 **Microsoft Agent Framework (MAF)** 를 이용해서
AI 에이전트를 직접 만들어 봅니다.

### 학습 목표
1. AI 에이전트가 무엇인지 코드로 직접 체험한다
2. 에이전트에 **도구(함수)** 를 붙여 능력을 확장한다
3. **여러 에이전트가 협업**하는 모습을 본다
4. 간단한 **워크플로우**를 직접 만들어본다

### 준비물
- Python 3.10 이상
- `pip install agent-framework`
- OpenAI 또는 Azure OpenAI API 키

---
## 0. 환경 준비하기

먼저 필요한 패키지를 설치하고, `.env.sample`을 복사해 APIM 정보를 채웁니다.

 > **TIP:** 키를 코드에 직접 적지 말고 `.env`로 관리하세요.  
 > 노트북에서는 `../.env`를 읽도록 설정합니다.  
 > `cp setup-apim/.env.sample ../.env` 후  
 > `../.env`의 `APIM_BASE_URL`, `APIM_KEY`, `CHAT_MODEL` 값을 채우면 됩니다.

In [4]:
# 패키지 설치 (이미 설치되어 있다면 생략 가능)
# !pip install agent-framework python-dotenv

import os

from dotenv import load_dotenv
from agent_framework.openai import OpenAIChatClient

load_dotenv("../.env", override=True)

def build_chat_client():
    apim_base_url = os.environ["APIM_BASE_URL"].rstrip("/")
    apim_key = os.environ["APIM_KEY"]
    model = os.getenv("CHAT_MODEL", "gpt-5.4")
    return OpenAIChatClient(
        model=model,
        base_url=f"{apim_base_url}/{model}/",
        api_key="placeholder",
        default_headers={"api-key": apim_key},
    )

# APIM 설정이 잘 되었는지 확인
if not os.environ.get("APIM_BASE_URL") or not os.environ.get("APIM_KEY"):
    print("⚠️  APIM 설정이 비어 있어요.")
    print("   1) cp setup-apim/.env.sample ../.env")
    print("   2) ../.env에 APIM_BASE_URL, APIM_KEY를 입력하세요.")
else:
    print("✅ APIM 설정이 준비되었습니다. 시작해 볼까요?")
    print("   CHAT_MODEL =", os.getenv("CHAT_MODEL", "gpt-5.4"))

✅ APIM 설정이 준비되었습니다. 시작해 볼까요?
   CHAT_MODEL = gpt-5.4


---
# 시나리오 1. Hello, Agent! 👋

**목표:** 첫 번째 AI 에이전트를 만들어서 인사를 시켜봅시다.

## 개념 정리
- **Agent**: 대화형 AI 에이전트의 기본 클래스
- **client**: 어떤 LLM(예: GPT-4)을 사용할지 정해주는 부분
- **instructions**: 에이전트의 "성격"이나 "역할"을 정해주는 시스템 메시지
- **name**: 에이전트의 이름 (선택)

In [3]:
from agent_framework import Agent

# 첫 번째 에이전트 만들기
greeter = Agent(
    client=build_chat_client(),
    name="인사봇",
    instructions=(
        "너는 친절한 인사 도우미야. "
        "항상 밝고 따뜻하게, 그리고 한국어로 대답해. "
        "이모지를 적절히 사용해도 좋아."
    ),
)

print("에이전트가 만들어졌어요!")
print(f"이름: {greeter.name}")

에이전트가 만들어졌어요!
이름: 인사봇


이제 에이전트에게 질문해 봅시다.

`agent.run("질문")` 으로 호출하면 답변이 돌아옵니다.
MAF는 `async` 함수이기 때문에 `await` 키워드를 같이 써야 해요.

In [ ]:
# 에이전트에게 말 걸기
response = await greeter.run("안녕! 너는 누구니?")
print(response.text)

In [ ]:
# 한 번 더 — 이번엔 자기소개를 부탁해 봅시다
response = await greeter.run("나는 고등학생이야. AI에 대해 배우고 있어!")
print(response.text)

### 🤔 생각해보기
- `instructions` 를 바꾸면 에이전트의 말투가 어떻게 달라질까요?
- 예: `"너는 시크한 고양이야. 짧고 도도하게 대답해."` 로 바꿔보세요.

---
# 시나리오 2. 도구를 쓰는 에이전트 🔧

**목표:** AI 에이전트에 **계산기**라는 도구를 붙여서,
직접 계산하지 못하는 LLM이 정확한 답을 낼 수 있게 만들어 봅시다.

## 왜 도구가 필요할까?
GPT 같은 언어 모델은 단어를 예측하는 모델이라서,
**복잡한 계산이나 최신 정보**를 모를 때가 많아요.
이때 "이런 일은 너 대신 이 함수를 써"라고 알려주면
에이전트가 그 함수(=도구)를 직접 호출해서 정답을 가져옵니다.

MAF에서는 그냥 **파이썬 함수**를 정의해서 에이전트에게 넘겨주면 끝!

In [ ]:
from typing import Annotated
from pydantic import Field

def add(
    a: Annotated[float, Field(description="첫 번째 숫자")],
    b: Annotated[float, Field(description="두 번째 숫자")],
) -> float:
    """두 숫자를 더한 값을 돌려준다."""
    return a + b

def multiply(
    a: Annotated[float, Field(description="첫 번째 숫자")],
    b: Annotated[float, Field(description="두 번째 숫자")],
) -> float:
    """두 숫자를 곱한 값을 돌려준다."""
    return a * b

def power(
    base: Annotated[float, Field(description="밑")],
    exp: Annotated[float, Field(description="지수")],
) -> float:
    """base의 exp 제곱을 돌려준다."""
    return base ** exp

print("세 가지 도구(add, multiply, power)를 만들었어요.")

이제 이 함수들을 에이전트에게 **도구**로 등록합니다.

In [ ]:
math_helper = Agent(
    client=build_chat_client(),
    name="수학도우미",
    instructions=(
        "너는 수학 문제를 도와주는 친절한 튜터야. "
        "계산이 필요하면 반드시 제공된 도구(add, multiply, power)를 사용해서 답을 구해. "
        "머릿속으로 계산하지 말고 도구를 호출해야 정확해."
    ),
    tools=[add, multiply, power],   # 👈 여기에 함수들을 넘겨요
)

response = await math_helper.run(
    "235 × 47은 얼마야? 그리고 그 결과를 제곱하면?"
)
print(response.text)

에이전트가 어떤 도구를 어떻게 호출했는지 살펴볼 수도 있어요.

In [ ]:
# 에이전트가 사용한 도구 호출 기록 보기
for msg in response.messages:
    if hasattr(msg, 'content_items'):
        for item in msg.content_items:
            print(type(item).__name__, '→', item)

### 🤔 생각해보기
- `divide(a, b)` 도구도 만들어서 추가해 보세요. (0으로 나눌 때 어떻게 처리?)
- 단위 변환 도구(℃ → ℉)를 만들면 에이전트가 "오늘 25도" 같은 질문에 답할 수 있어요.

---
# 시나리오 3. 두 에이전트의 협업 ✍️

**목표:** 서로 다른 역할을 가진 두 AI가 협력해서 글을 쓰는 모습을 봅시다.

## 시나리오
- **작가 에이전트**: 짧은 글을 씁니다
- **편집자 에이전트**: 그 글을 읽고 수정 의견을 줍니다
- 둘이 한 번씩 주고받으며 글의 품질을 높입니다

여러 에이전트를 만들 때는 그냥 `Agent`를 여러 개 만들고
한 에이전트의 결과를 다른 에이전트에게 입력으로 넘기면 됩니다.

In [ ]:
# 1. 작가 에이전트
writer = Agent(
    client=build_chat_client(),
    name="작가",
    instructions=(
        "너는 감성적인 단편 작가야. "
        "주제를 받으면 한국어로 4~5문장의 짧은 글을 써. "
        "비유와 묘사를 풍부하게 사용해."
    ),
)

# 2. 편집자 에이전트
editor = Agent(
    client=build_chat_client(),
    name="편집자",
    instructions=(
        "너는 꼼꼼한 편집자야. "
        "받은 글을 읽고, 더 좋아질 수 있는 부분 2~3가지를 짚어줘. "
        "건설적이고 친절한 어조로 피드백해."
    ),
)

print("두 명의 에이전트가 준비됐어요: 작가, 편집자")

In [ ]:
# 작가가 먼저 글을 씁니다
topic = "비 오는 날의 등굣길"
draft = await writer.run(f"주제: {topic}. 이 주제로 짧은 글을 써줘.")
print("📝 [작가의 초안]")
print(draft.text)
print()

In [ ]:
# 편집자가 피드백합니다
feedback = await editor.run(
    f"아래는 작가가 쓴 글이야. 어떻게 더 좋아질 수 있을지 의견을 줘.\n\n{draft.text}"
)
print("🧐 [편집자의 피드백]")
print(feedback.text)
print()

In [ ]:
# 작가가 피드백을 받아 다시 씁니다
final = await writer.run(
    f"네가 쓴 글에 대한 편집자의 피드백이야:\n{feedback.text}\n\n"
    f"이 피드백을 반영해서 글을 다시 써줘."
)
print("✨ [작가의 최종본]")
print(final.text)

### 🤔 생각해보기
- 편집자에게 "3번 라운드까지 반복하자" 같은 규칙을 줘 보면 어떨까요?
- 사실 검증을 담당하는 **팩트체커 에이전트**를 추가해도 재미있어요.

---
# 시나리오 4. 워크플로우 만들기 🔗

**목표:** 여러 에이전트를 **그래프(워크플로우)** 로 연결해서
자동으로 단계별로 동작하게 만들어 봅시다.

## 워크플로우란?
- 시나리오 3처럼 매번 손으로 결과를 넘겨주는 대신
- 미리 **"누가 → 누구에게 → 어떻게"** 연결할지 정의해 두는 것
- MAF의 `WorkflowBuilder`를 쓰면 시각화도 되고 디버깅도 쉬워요

## 이번 워크플로우
사용자 질문 → **요약 에이전트** (한 줄로 정리) → **번역 에이전트** (영어로 번역) → 결과

> ⚠️ 워크플로우 API는 MAF 버전에 따라 조금씩 달라질 수 있어요.
> 아래는 가장 자주 쓰이는 패턴입니다.

In [ ]:
from agent_framework import WorkflowBuilder, AgentExecutor

# 1. 두 개의 에이전트 만들기
summarizer = Agent(
    client=build_chat_client(),
    name="요약가",
    instructions="긴 문장을 한 줄로 핵심만 요약해. 한국어로.",
)

translator = Agent(
    client=build_chat_client(),
    name="번역가",
    instructions="입력 받은 한국어 문장을 자연스러운 영어로 번역해.",
)

# 2. 워크플로우 만들기
summarize_step = AgentExecutor(summarizer, id="summarize")
translate_step = AgentExecutor(translator, id="translate")

workflow = (
    WorkflowBuilder()
    .set_start_executor(summarize_step)
    .add_edge(summarize_step, translate_step)
    .build()
)

print("워크플로우가 만들어졌어요!")
print("흐름: 입력 → 요약가 → 번역가 → 출력")

In [ ]:
# 3. 워크플로우 실행
long_text = (
    "오늘 학교에서 과학 수업 시간에 인공지능에 대해 배웠다. "
    "선생님께서는 AI가 단순히 챗봇이 아니라 스스로 도구를 사용하고 "
    "여러 단계를 거쳐 일을 해내는 에이전트로 발전하고 있다고 하셨다. "
    "수업이 끝나고 친구들과 함께 다음 주에 직접 만들어 보기로 했다."
)

events = await workflow.run(long_text)

# 결과 출력
print("📥 [입력]")
print(long_text)
print()
print("📤 [최종 결과]")
print(events.get_outputs())

### 🤔 생각해보기
- 번역 다음에 **분위기 분석 에이전트**를 한 단계 더 추가해 보세요.
- 어떤 조건일 때만 한쪽 길로 가는 **분기(branch)** 도 만들 수 있어요.

---
# 🎓 마무리 & 다음 단계

오늘 우리는:

1. **첫 에이전트** 를 만들어 인사를 시켜 봤고  
2. **도구(함수)** 를 붙여 계산을 정확하게 시켰고  
3. **두 에이전트** 가 작가-편집자로 협업하게 했고  
4. **워크플로우** 로 자동 파이프라인을 만들었습니다.

## 응용 아이디어 💡
- **나만의 학습 도우미**: 단어 퀴즈 에이전트 + 채점 에이전트
- **여행 플래너**: 검색 에이전트 + 일정 정리 에이전트 + 예산 계산 에이전트
- **창작 도우미**: 시놉시스 작가 + 등장인물 디자이너 + 일러스트 프롬프트 작성가
- **뉴스 브리핑**: 기사 수집기 + 요약가 + 음성 변환기

## 더 알아보기 📚
- 공식 문서: <https://learn.microsoft.com/agent-framework/>
- GitHub: <https://github.com/microsoft/agent-framework>
- Python 패키지: `pip install agent-framework`

**기억하세요:** 좋은 AI 에이전트는 화려한 코드가 아니라
**좋은 instructions**(역할/규칙)과 **잘 설계된 도구**에서 나옵니다.  
오늘 배운 걸로 여러분만의 작은 AI를 만들어 보세요! 🚀